In [1]:
import pandas as pd


In [2]:
pd.__version__

'1.4.2'

In [2]:
!pip install pyarrow


In [4]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')


In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [3]:
import sklearn


In [7]:
sklearn.__version__

'1.0.2'

In [8]:
print(df.shape)
print(df.columns.tolist())

(3066766, 19)
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


Q1 Answer : 19 Columns

In [9]:
df['duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

print(df['duration'].std())

42.594351241920904


Q2: 42.5

In [10]:
n_before = len(df)

df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

n_after = len(df)
fraction = n_after / n_before
print(f"{fraction:.4f} ->  {fraction * 100:.1f}%")

0.9812 ->  98.1%


fraction of the records left after you dropped the outliers: 98.1%

In [ ]:
from sklearn.feature_extraction import DictVectorizer

df[['PULocationID', 'DOLocationID']] = df[['PULocationID', 'DOLocationID']].astype(str)

train_dicts = df[['PULocationID', 'DOLocationID']].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

print(X_train.shape)


In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer

# Load
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')

# Duration
df['duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

# Drop outliers
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

# Keep only needed columns to free memory
df = df[['PULocationID', 'DOLocationID', 'duration']]

# One-hot encode
df[['PULocationID', 'DOLocationID']] = df[['PULocationID', 'DOLocationID']].astype(str)
train_dicts = df[['PULocationID', 'DOLocationID']].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

print(X_train.shape)


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

y_train = df['duration'].values

lr = LinearRegression()
lr.fit(X_train, y_train)

rmse = np.sqrt(mean_squared_error(y_train, lr.predict(X_train)))
print(f"RMSE on train: {rmse:.2f}")

(3009173, 515)
RMSE on train: 7.65


Number of columns does the matrix have: 515

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# --- January (train) ---
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df['duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
df = df[['PULocationID', 'DOLocationID', 'duration']]
df[['PULocationID', 'DOLocationID']] = df[['PULocationID', 'DOLocationID']].astype(str)

dv = DictVectorizer()
X_train = dv.fit_transform(df[['PULocationID', 'DOLocationID']].to_dict(orient='records'))
y_train = df['duration'].values

lr = LinearRegression()
lr.fit(X_train, y_train)
print(f"Q5 RMSE train: {np.sqrt(mean_squared_error(y_train, lr.predict(X_train))):.2f}")

# --- February (validation) ---
df_feb = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')
df_feb['duration'] = (df_feb['tpep_dropoff_datetime'] - df_feb['tpep_pickup_datetime']).dt.total_seconds() / 60
df_feb = df_feb[(df_feb['duration'] >= 1) & (df_feb['duration'] <= 60)]
df_feb = df_feb[['PULocationID', 'DOLocationID', 'duration']]
df_feb[['PULocationID', 'DOLocationID']] = df_feb[['PULocationID', 'DOLocationID']].astype(str)

X_val = dv.transform(df_feb[['PULocationID', 'DOLocationID']].to_dict(orient='records'))
y_val = df_feb['duration'].values
print(f"Q6 RMSE validation: {np.sqrt(mean_squared_error(y_val, lr.predict(X_val))):.2f}")

Q5 RMSE train: 7.65
